# Stage 3 — Expert-Performance-Aware Multimodal MoE

This notebook extends the Stage 2 confidence-aware system with a **task-specific expert-performance predictor**. In addition to deciding whether a request is classification, segmentation, or ambiguous, the system estimates whether the selected YOLO expert is likely to meet a task-specific success criterion.

The Stage 1 YOLO experts and Stage 2 routing gate are reused. A new quality head is trained from observed expert outcomes. At inference time, the system can abstain when the Stage 2 gate is uncertain **or** when the selected expert is predicted unlikely to succeed.

> **Important design distinction:** the classification and segmentation experts solve different tasks, so their scores are not directly comparable. This stage does not choose between them for the same task. It predicts the expected success of the expert that matches the requested task, then uses that estimate as an additional accept/review signal.

## Learning objectives

- Measure expert outcomes against ground-truth labels rather than relying only on raw YOLO confidence.
- Create task-specific success labels: classification correctness and segmentation IoU.
- Train an auxiliary performance head while keeping CLIP and the Stage 2 task router frozen.
- Evaluate quality prediction with Brier score, calibration error, and threshold/coverage analysis.
- Combine task routing, Stage 2 abstention, and predicted expert success in one inference policy.

## Pipeline

```mermaid
flowchart TD
    A[Image + Text Prompt] --> B[CLIP Multimodal Features]
    B --> C[Stage 2 Task Gate]
    B --> D[Expert-Performance Head]
    C --> E{Task / Ambiguity}
    E -->|Classification| F[YOLO Classification]
    E -->|Segmentation| G[YOLO Segmentation]
    E -->|Ambiguous or low routing confidence| H[Clarify / Abstain]
    D --> I{Predicted success for selected task}
    I -->|Above quality threshold| J[Dispatch to selected expert]
    I -->|Below quality threshold| K[Abstain / Human Review]
```

## Before you start

Run Stage 1 and Stage 2 first. This notebook expects the following files in the same `artifacts` directory created by those notebooks:

- `router_confidence_aware.pt`
- `moe_confidence_config.json`
- `classification_best.pt`
- `segmentation_best.pt`

The notebook uses Oxford-IIIT Pet, matching the earlier stages. Expert outcome labels are generated on a held-out source-image subset. Stage 1 used the Oxford test split for expert checkpoint selection, and Stage 2 used it for calibration/evaluation; therefore, results here are a **progressive learning demonstration**, not a pristine independent benchmark.

## 1. Environment, operating system and compute device

This cell installs the required libraries, detects CUDA / Apple MPS / CPU, and sets portable paths. On Colab, Google Drive is used by default so checkpoints persist after the session ends.

In [ ]:
%pip install -q open_clip_torch ultralytics opencv-python-headless torchvision matplotlib pillow

import os
import sys
import json
import math
import random
import platform
from pathlib import Path
from typing import Any, Dict, List

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if IN_COLAB:
    from google.colab import drive
    MOUNT_GOOGLE_DRIVE = True
    if MOUNT_GOOGLE_DRIVE:
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive/yolo_moe_project")
    else:
        PROJECT_DIR = Path("/content/yolo_moe_project")
else:
    PROJECT_DIR = Path.cwd() / "yolo_moe_project"

DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
for folder in (PROJECT_DIR, DATA_DIR, ARTIFACT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"

def get_ultralytics_device(device: torch.device):
    """Convert a PyTorch device to an Ultralytics device argument.

    Input: torch.device (CUDA, MPS, or CPU).
    Output: 0 for the first CUDA GPU, 'mps' for Apple Silicon, or 'cpu'.
    """
    if device.type == "cuda":
        return 0
    if device.type == "mps":
        return "mps"
    return "cpu"

YOLO_DEVICE = get_ultralytics_device(DEVICE)
print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Colab:", IN_COLAB)
print("Device:", DEVICE)
print("Project directory:", PROJECT_DIR)
print("Artifact directory:", ARTIFACT_DIR)

## 2. Check Stage 1 and Stage 2 artifacts

The Stage 2 checkpoint supplies the three-way task/clarification gate. The two Stage 1 YOLO checkpoints supply the actual task experts. This notebook does not retrain those experts.

In [ ]:
STAGE2_ROUTER_PATH = ARTIFACT_DIR / "router_confidence_aware.pt"
STAGE2_CONFIG_PATH = ARTIFACT_DIR / "moe_confidence_config.json"
CLS_WEIGHTS_PATH = ARTIFACT_DIR / "classification_best.pt"
SEG_WEIGHTS_PATH = ARTIFACT_DIR / "segmentation_best.pt"

required_files = [STAGE2_ROUTER_PATH, STAGE2_CONFIG_PATH,
                  CLS_WEIGHTS_PATH, SEG_WEIGHTS_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing Stage 1/2 artifacts:\\n" + "\\n".join(missing_files) +
        "\\nComplete Stage 1 and Stage 2, then place their artifacts in ARTIFACT_DIR."
    )

stage2_checkpoint = torch.load(STAGE2_ROUTER_PATH, map_location="cpu")
stage2_config = json.loads(STAGE2_CONFIG_PATH.read_text(encoding="utf-8"))
STAGE2_TEMPERATURE = float(stage2_checkpoint.get("temperature", stage2_config.get("temperature", 1.0)))
STAGE2_ROUTING_THRESHOLD = float(stage2_checkpoint.get(
    "routing_threshold", stage2_config.get("routing_threshold", 0.70)
))
print("All required checkpoints found.")
print("Stage 2 temperature:", STAGE2_TEMPERATURE)
print("Stage 2 routing threshold:", STAGE2_ROUTING_THRESHOLD)

## 3. Define the expert-performance targets

The quality head predicts the probability that the selected expert meets a clear, task-specific success criterion. These criteria are deliberately separate because the two tasks have different metrics.

In [ ]:
DECISION_CLASSIFICATION = 0
DECISION_SEGMENTATION = 1
DECISION_CLARIFY = 2
DECISION_NAMES = {
    DECISION_CLASSIFICATION: "classification",
    DECISION_SEGMENTATION: "segmentation",
    DECISION_CLARIFY: "clarify",
}

# A classification example is successful when the top-1 breed is correct.
# A segmentation example is successful when foreground IoU reaches this value.
SEGMENTATION_SUCCESS_IOU = 0.50

# Inference abstains / requests review when predicted success probability is lower.
QUALITY_THRESHOLD = 0.60

# Limit the number of held-out source images used to create outcome labels.
# Increase this for a larger experiment; expert inference time will increase.
MAX_QUALITY_IMAGES = 600
EXPERT_INFERENCE_BATCH = 16

print("Classification success: top-1 correct")
print("Segmentation success: IoU >=", SEGMENTATION_SUCCESS_IOU)
print("Initial predicted-success threshold:", QUALITY_THRESHOLD)
print("Maximum source images:", MAX_QUALITY_IMAGES)

## 4. Prepare a source-image split

Each source image is assigned to either the quality-head training subset or the held-out evaluation subset. Both task prompts are generated for each image. The split is performed before creating prompt/task rows so that variants of one image cannot leak across the two subsets.

In [ ]:
quality_base = datasets.OxfordIIITPet(
    root=str(DATA_DIR), split="test",
    target_types=["category", "segmentation"], download=True
)

all_indices = list(range(len(quality_base)))
random.Random(SEED + 300).shuffle(all_indices)
if MAX_QUALITY_IMAGES is not None:
    all_indices = all_indices[:min(MAX_QUALITY_IMAGES, len(all_indices))]

split_point = max(1, int(0.5 * len(all_indices)))
quality_train_indices = all_indices[:split_point]
quality_eval_indices = all_indices[split_point:]
if not quality_eval_indices:
    raise ValueError("Need at least two source images to create train/evaluation subsets.")

print("Available test images:", len(quality_base))
print("Quality-head training images:", len(quality_train_indices))
print("Held-out quality evaluation images:", len(quality_eval_indices))

## 5. Run the two experts and create ground-truth outcome labels

This is the most compute-intensive step. The classification expert is scored by top-1 correctness. The segmentation expert is scored by foreground intersection-over-union (IoU) against the Oxford trimap foreground (labels 1 and 3, matching the Stage 1 export setting). No raw YOLO confidence score is treated as ground-truth quality.

In [ ]:
from ultralytics import YOLO

cls_expert = YOLO(str(CLS_WEIGHTS_PATH))
seg_expert = YOLO(str(SEG_WEIGHTS_PATH))


def segmentation_iou_from_result(result, ground_truth_mask: np.ndarray) -> float:
    """Compute foreground IoU for one Ultralytics segmentation result.

    Inputs:
        result: Ultralytics Results for one image.
        ground_truth_mask: 2-D boolean/0-1 array; foreground is non-zero.
    Output:
        IoU in [0, 1]. If no predicted mask is returned, IoU is zero.
    """
    gt = (ground_truth_mask > 0).astype(np.uint8)
    if result.masks is None or result.masks.data is None or len(result.masks.data) == 0:
        pred = np.zeros_like(gt, dtype=np.uint8)
    else:
        masks = result.masks.data.detach().float().cpu().numpy()
        combined = masks.max(axis=0) > 0.5
        pred = cv2.resize(
            combined.astype(np.uint8), (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )
    intersection = int(np.logical_and(pred > 0, gt > 0).sum())
    union = int(np.logical_or(pred > 0, gt > 0).sum())
    return float(intersection / union) if union else 1.0


def create_expert_outcome_records(dataset, source_indices: List[int]) -> List[Dict[str, Any]]:
    """Run both experts and build per-image performance targets.

    Inputs:
        dataset: Oxford-IIIT Pet dataset with category and segmentation targets.
        source_indices: source image indices to score.
    Output:
        One record per source image containing the RGB PIL image, class label,
        classification correctness, segmentation IoU, and segmentation success.
    """
    images, metadata = [], []
    for source_idx in source_indices:
        image, (category_id, trimap) = dataset[source_idx]
        image = image.convert("RGB")
        trimap_array = np.asarray(trimap, dtype=np.uint8)
        # Stage 1 segmentation export included both pet (1) and boundary (3).
        gt_foreground = (trimap_array == 1) | (trimap_array == 3)
        images.append(image)
        metadata.append({
            "source_index": int(source_idx),
            "image": image,
            "class_label": int(category_id) - 1,
            "gt_foreground": gt_foreground,
        })

    cls_results = cls_expert.predict(
        source=images, batch=EXPERT_INFERENCE_BATCH, imgsz=224,
        device=YOLO_DEVICE, verbose=False
    )
    seg_results = seg_expert.predict(
        source=images, batch=EXPERT_INFERENCE_BATCH, imgsz=640,
        device=YOLO_DEVICE, verbose=False
    )
    if len(cls_results) != len(metadata) or len(seg_results) != len(metadata):
        raise RuntimeError("Ultralytics returned a different number of results than input images.")

    records = []
    for meta, cls_result, seg_result in zip(metadata, cls_results, seg_results):
        if cls_result.probs is None:
            raise RuntimeError("Classification expert returned no class probabilities.")
        predicted_class = int(cls_result.probs.top1)
        cls_success = int(predicted_class == meta["class_label"])
        seg_iou = segmentation_iou_from_result(seg_result, meta["gt_foreground"])
        seg_success = int(seg_iou >= SEGMENTATION_SUCCESS_IOU)
        records.append({
            "source_index": meta["source_index"],
            "image": meta["image"],
            "class_label": meta["class_label"],
            "classification_success": cls_success,
            "segmentation_iou": seg_iou,
            "segmentation_success": seg_success,
        })
    return records

print("Expert models loaded. Starting outcome-label generation...")
train_outcome_records = create_expert_outcome_records(quality_base, quality_train_indices)
eval_outcome_records = create_expert_outcome_records(quality_base, quality_eval_indices)
print("Training records:", len(train_outcome_records))
print("Evaluation records:", len(eval_outcome_records))

## 6. Inspect the observed expert outcomes

These metrics describe the experts on the selected source-image subsets. They are not directly comparable across tasks: classification accuracy and segmentation IoU measure different things.

In [ ]:
def summarize_outcomes(records, split_name):
    """Print task-specific expert outcome statistics for a record collection."""
    cls_acc = np.mean([r["classification_success"] for r in records])
    seg_ious = np.asarray([r["segmentation_iou"] for r in records], dtype=float)
    seg_success = np.mean([r["segmentation_success"] for r in records])
    print(f"{split_name} | images={len(records)}")
    print(f"  Classification top-1 accuracy: {cls_acc:.3f}")
    print(f"  Segmentation mean IoU:         {seg_ious.mean():.3f}")
    print(f"  Segmentation success rate:     {seg_success:.3f} (IoU >= {SEGMENTATION_SUCCESS_IOU:.2f})")

summarize_outcomes(train_outcome_records, "Quality-head training subset")
summarize_outcomes(eval_outcome_records, "Held-out evaluation subset")

## 7. Build the multimodal quality-prediction dataset

Each source image produces two examples: one classification prompt and one segmentation prompt. The target is the observed success/failure of the corresponding expert. The quality head has two outputs, but the loss updates only the output associated with the requested task.

In [ ]:
CLASSIFICATION_PROMPTS = [
    "Classify the pet breed in this image.",
    "Identify the breed of the animal shown.",
    "What pet breed is visible in this picture?",
    "Return the breed category for this pet.",
]
SEGMENTATION_PROMPTS = [
    "Segment the pet from the background.",
    "Create a pixel mask around the animal.",
    "Outline the pet and separate it from the background.",
    "Return a segmentation mask for the animal.",
]

class ExpertPerformanceDataset(Dataset):
    """Pair image/task prompts with observed expert-success labels.

    Inputs:
        records: records returned by create_expert_outcome_records().
        image_size: square CLIP router input size.
        seed: prompt-selection seed.
    Outputs from __getitem__:
        image: RGB float tensor [3,H,W] in [0,1].
        prompt: task instruction string.
        task_id: 0 for classification, 1 for segmentation.
        success: binary observed outcome for the requested expert.
    """
    def __init__(self, records, image_size=224, seed=42):
        self.records = records
        self.image_size = image_size
        self.rng = random.Random(seed)
        self.samples = [(idx, task_id) for idx in range(len(records)) for task_id in (0, 1)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        """Return one image/prompt/task/success training example."""
        record_idx, task_id = self.samples[index]
        record = self.records[record_idx]
        image = record["image"].resize(
            (self.image_size, self.image_size), Image.Resampling.BICUBIC
        )
        array = np.asarray(image, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()
        prompt_pool = CLASSIFICATION_PROMPTS if task_id == 0 else SEGMENTATION_PROMPTS
        prompt = self.rng.choice(prompt_pool)
        success = record["classification_success"] if task_id == 0 else record["segmentation_success"]
        return {
            "image": image_tensor,
            "prompt": prompt,
            "task_id": torch.tensor(task_id, dtype=torch.long),
            "success": torch.tensor(success, dtype=torch.float32),
        }


def collate_performance_batch(batch):
    """Stack image and scalar targets while keeping prompts as a string list."""
    return {
        "images": torch.stack([item["image"] for item in batch]),
        "prompts": [item["prompt"] for item in batch],
        "task_ids": torch.stack([item["task_id"] for item in batch]),
        "success": torch.stack([item["success"] for item in batch]),
    }

train_quality_dataset = ExpertPerformanceDataset(train_outcome_records, seed=SEED + 1)
eval_quality_dataset = ExpertPerformanceDataset(eval_outcome_records, seed=SEED + 2)
train_quality_loader = DataLoader(
    train_quality_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY, collate_fn=collate_performance_batch
)
eval_quality_loader = DataLoader(
    eval_quality_dataset, batch_size=32, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY, collate_fn=collate_performance_batch
)
print("Training task examples:", len(train_quality_dataset))
print("Evaluation task examples:", len(eval_quality_dataset))

## 8. Define the Stage 3 model and load the Stage 2 gate

The model retains the Stage 2 three-way routing head and adds a separate two-output expert-performance head. The Stage 2 weights initialize the original components. CLIP and the Stage 2 task gate remain frozen; only the new performance head is trained in this notebook.

In [ ]:
import open_clip

class CLIPPerformanceAwareRouter(nn.Module):
    """Stage 2 CLIP router augmented with task-specific expert-success logits.

    Inputs to forward():
        images: RGB float tensor [B,3,224,224] in [0,1].
        prompts: list[str] with B natural-language task instructions.
    Outputs:
        dict with routing logits [B,3] and expert-success logits [B,2].
        Output 0 estimates classification-expert success; output 1 estimates
        segmentation-expert success. These are not cross-task quality rankings.
    """
    def __init__(self, clip_model_name="ViT-B-32",
                 clip_pretrained="laion2b_s34b_b79k", hidden_dim=256, dropout=0.1):
        super().__init__()
        self.clip_model_name = clip_model_name
        self.clip_pretrained = clip_pretrained
        self.clip_model, _, _ = open_clip.create_model_and_transforms(
            model_name=clip_model_name, pretrained=clip_pretrained
        )
        self.tokenizer = open_clip.get_tokenizer(clip_model_name)
        self.register_buffer("clip_mean", torch.tensor(
            [0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1))
        self.register_buffer("clip_std", torch.tensor(
            [0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1))
        for parameter in self.clip_model.parameters():
            parameter.requires_grad = False
        self.clip_model.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            feature_dim = self.clip_model.encode_image(
                (dummy - self.clip_mean) / self.clip_std
            ).shape[-1]
        fused_dim = feature_dim * 3
        self.gate = nn.Sequential(
            nn.Linear(fused_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3),
        )
        self.quality_head = nn.Sequential(
            nn.Linear(fused_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def train(self, mode=True):
        """Set module mode while keeping the frozen CLIP encoder in evaluation mode."""
        super().train(mode)
        self.clip_model.eval()
        return self

    def forward(self, images, prompts):
        """Return task-routing logits and task-specific expert-success logits."""
        images = (images - self.clip_mean.to(images.device)) / self.clip_std.to(images.device)
        tokens = self.tokenizer(prompts).to(images.device)
        with torch.no_grad():
            image_features = self.clip_model.encode_image(images)
            text_features = self.clip_model.encode_text(tokens)
        image_features = F.normalize(image_features.float(), dim=-1)
        text_features = F.normalize(text_features.float(), dim=-1)
        fused = torch.cat(
            [image_features, text_features, image_features * text_features], dim=-1
        )
        return {
            "routing_logits": self.gate(fused),
            "quality_logits": self.quality_head(fused),
        }

router = CLIPPerformanceAwareRouter(
    clip_model_name=stage2_checkpoint["clip_model_name"],
    clip_pretrained=stage2_checkpoint["clip_pretrained"],
).to(DEVICE)
load_result = router.load_state_dict(stage2_checkpoint["model_state_dict"], strict=False)
print("Loaded Stage 2 parameters.")
print("Missing keys (expected: new quality_head only):", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# Freeze the original Stage 2 components. Only the new quality head is optimized.
for parameter in router.clip_model.parameters():
    parameter.requires_grad = False
for parameter in router.gate.parameters():
    parameter.requires_grad = False
for parameter in router.quality_head.parameters():
    parameter.requires_grad = True

trainable_parameters = sum(p.numel() for p in router.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_parameters)

## 9. Train the expert-performance head

Binary cross-entropy is applied only to the requested task output. For a classification prompt, the target is whether top-1 classification was correct. For a segmentation prompt, the target is whether IoU met the configured threshold. The Stage 2 task router is not changed.

In [ ]:
def run_quality_epoch(model, loader, optimizer=None, device=DEVICE):
    """Run one epoch of task-specific expert-success prediction.

    Inputs:
        model: CLIPPerformanceAwareRouter.
        loader: batches from collate_performance_batch().
        optimizer: optimizer for training; None for evaluation.
        device: torch device.
    Output:
        dictionary containing mean BCE loss and binary accuracy for the
        quality output associated with each example's task.
    """
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total = 0.0, 0, 0
    grad_context = torch.enable_grad if training else torch.no_grad
    for batch in loader:
        images = batch["images"].to(device, non_blocking=True)
        task_ids = batch["task_ids"].to(device, non_blocking=True)
        targets = batch["success"].to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with grad_context():
            output = model(images, batch["prompts"])
            selected_logits = output["quality_logits"].gather(
                1, task_ids.unsqueeze(1)
            ).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(selected_logits, targets)
            if training:
                loss.backward()
                optimizer.step()
        predictions = (torch.sigmoid(selected_logits) >= 0.5).float()
        count = targets.numel()
        total_loss += loss.item() * count
        total_correct += int((predictions == targets).sum().item())
        total += count
    return {
        "loss": total_loss / max(total, 1),
        "accuracy": total_correct / max(total, 1),
    }

QUALITY_EPOCHS = 12
QUALITY_LEARNING_RATE = 1e-3
optimizer = torch.optim.AdamW(
    router.quality_head.parameters(), lr=QUALITY_LEARNING_RATE, weight_decay=1e-4
)

history = []
for epoch in range(1, QUALITY_EPOCHS + 1):
    train_metrics = run_quality_epoch(router, train_quality_loader, optimizer, DEVICE)
    eval_metrics = run_quality_epoch(router, eval_quality_loader, None, DEVICE)
    history.append({"epoch": epoch, "train": train_metrics, "evaluation": eval_metrics})
    print(
        f"Epoch {epoch:02d}/{QUALITY_EPOCHS} | "
        f"train BCE {train_metrics['loss']:.4f}, acc {train_metrics['accuracy']:.3f} | "
        f"held-out BCE {eval_metrics['loss']:.4f}, acc {eval_metrics['accuracy']:.3f}"
    )

## 10. Collect held-out expert-success predictions

The evaluation loader is not used to update the quality head. We collect the predicted success probability for each task example and compare it with the measured expert outcome.

In [ ]:
@torch.no_grad()
def collect_quality_predictions(model, loader, device=DEVICE):
    """Collect task-specific quality probabilities and observed binary targets.

    Inputs: trained router, evaluation DataLoader, and device.
    Outputs: CPU tensors for predicted success probabilities, labels, and task IDs.
    Uses no_grad (not inference_mode) so the returned tensors remain ordinary
    tensors that can safely be used in later tensor operations if required.
    """
    model.eval()
    probabilities, labels, task_ids_all = [], [], []
    for batch in loader:
        images = batch["images"].to(device)
        output = model(images, batch["prompts"])
        selected_logits = output["quality_logits"].gather(
            1, batch["task_ids"].to(device).unsqueeze(1)
        ).squeeze(1)
        probabilities.append(torch.sigmoid(selected_logits).cpu())
        labels.append(batch["success"].cpu())
        task_ids_all.append(batch["task_ids"].cpu())
    return (
        torch.cat(probabilities),
        torch.cat(labels),
        torch.cat(task_ids_all),
    )

eval_quality_probs, eval_quality_labels, eval_task_ids = collect_quality_predictions(
    router, eval_quality_loader, DEVICE
)
print("Held-out quality predictions:", len(eval_quality_probs))

## 11. Evaluate quality prediction

Brier score measures the squared error of predicted probabilities. Expected Calibration Error (ECE) summarizes the gap between average confidence and observed success frequency across probability bins. Metrics are reported separately for classification and segmentation.

In [ ]:
def binary_ece(probabilities, labels, bins=10):
    """Compute expected calibration error for binary success probabilities.

    Inputs: 1-D probability and binary-label tensors, plus number of bins.
    Output: scalar ECE in [0,1]. Empty bins are ignored.
    """
    probabilities = probabilities.float().flatten()
    labels = labels.float().flatten()
    edges = torch.linspace(0.0, 1.0, bins + 1)
    ece = torch.tensor(0.0)
    for index in range(bins):
        lower, upper = edges[index], edges[index + 1]
        in_bin = (probabilities >= lower) & (
            probabilities <= upper if index == bins - 1 else probabilities < upper
        )
        if in_bin.any():
            confidence = probabilities[in_bin].mean()
            frequency = labels[in_bin].mean()
            ece += in_bin.float().mean() * torch.abs(confidence - frequency)
    return float(ece.item())


def quality_metrics(probabilities, labels, threshold=0.5):
    """Return Brier score, ECE, threshold accuracy, and base success rate."""
    probabilities = probabilities.float()
    labels = labels.float()
    predictions = (probabilities >= threshold).float()
    return {
        "brier_score": float(((probabilities - labels) ** 2).mean().item()),
        "ece": binary_ece(probabilities, labels),
        "accuracy_at_0.5": float((predictions == labels).float().mean().item()),
        "observed_success_rate": float(labels.mean().item()),
        "mean_predicted_success": float(probabilities.mean().item()),
    }

for task_id, task_name in [(0, "classification"), (1, "segmentation")]:
    selected = eval_task_ids == task_id
    print(task_name, quality_metrics(
        eval_quality_probs[selected], eval_quality_labels[selected]
    ))

## 12. Plot reliability diagrams

A well-calibrated quality predictor should have an observed success frequency close to its predicted probability. With a small demonstration subset, bins may contain few examples; interpret the plot cautiously.

In [ ]:
def plot_quality_reliability(probabilities, labels, title, bins=10):
    """Plot mean predicted success against empirical success frequency."""
    probabilities = probabilities.float()
    labels = labels.float()
    edges = torch.linspace(0.0, 1.0, bins + 1)
    mean_confidence, observed_frequency = [], []
    for index in range(bins):
        in_bin = (probabilities >= edges[index]) & (
            probabilities <= edges[index + 1] if index == bins - 1
            else probabilities < edges[index + 1]
        )
        if in_bin.any():
            mean_confidence.append(float(probabilities[in_bin].mean()))
            observed_frequency.append(float(labels[in_bin].mean()))
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.plot(mean_confidence, observed_frequency, marker="o", label="Quality head")
    plt.xlabel("Mean predicted probability of expert success")
    plt.ylabel("Observed expert success frequency")
    plt.title(title)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

for task_id, task_name in [(0, "Classification"), (1, "Segmentation")]:
    selected = eval_task_ids == task_id
    plot_quality_reliability(
        eval_quality_probs[selected], eval_quality_labels[selected],
        f"{task_name} expert-success reliability"
    )

## 13. Choose a quality threshold and inspect coverage

Coverage is the fraction of task requests accepted by the quality threshold. Accepted success rate is the observed success frequency among accepted requests. Raising the threshold generally reduces coverage and may increase accepted success; inspect the trade-off instead of assuming one threshold is universally best.

In [ ]:
def selective_quality_metrics(probabilities, labels, threshold):
    """Measure acceptance coverage and observed success among accepted examples."""
    accepted = probabilities >= threshold
    coverage = float(accepted.float().mean().item())
    accepted_success = (
        float(labels[accepted].mean().item()) if accepted.any() else float("nan")
    )
    return {
        "threshold": float(threshold),
        "coverage": coverage,
        "accepted_success_rate": accepted_success,
        "accepted_count": int(accepted.sum().item()),
        "total_count": int(labels.numel()),
    }

for task_id, task_name in [(0, "classification"), (1, "segmentation")]:
    selected = eval_task_ids == task_id
    print("\\n", task_name)
    for threshold in [0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
        print(selective_quality_metrics(
            eval_quality_probs[selected], eval_quality_labels[selected], threshold
        ))

print("\\nConfigured quality threshold:", QUALITY_THRESHOLD)

## 14. Assemble the Stage 3 inference system

The inference policy applies two sequential checks: (1) Stage 2 task confidence/ambiguity, then (2) predicted success for the expert associated with the selected task. A low predicted success probability results in abstention/review; the system does not silently switch to the other task expert.

In [ ]:
class PerformanceAwareYOLOMoE:
    """Inference wrapper combining Stage 2 routing and Stage 3 quality prediction.

    Constructor inputs:
        router: trained CLIPPerformanceAwareRouter.
        cls_weights_path / seg_weights_path: Stage 1 YOLO checkpoints.
        temperature: Stage 2 routing temperature.
        routing_threshold: minimum Stage 2 task confidence.
        quality_threshold: minimum predicted success probability for dispatch.
        device: PyTorch device.

    predict() inputs:
        image: path, PIL image, or RGB float tensor [3,H,W] in [0,1].
        prompt: natural-language instruction.
        yolo_conf: Ultralytics prediction confidence threshold.

    Output:
        Dictionary with status, route, routing probabilities, predicted expert
        success probability, review reason, and raw Ultralytics result (if run).
    """
    def __init__(self, router, cls_weights_path, seg_weights_path,
                 temperature=1.0, routing_threshold=0.70,
                 quality_threshold=0.60, device=DEVICE):
        self.device = torch.device(device)
        self.router = router.to(self.device).eval()
        self.cls_expert = YOLO(str(cls_weights_path))
        self.seg_expert = YOLO(str(seg_weights_path))
        self.temperature = float(temperature)
        self.routing_threshold = float(routing_threshold)
        self.quality_threshold = float(quality_threshold)

    @staticmethod
    def _to_pil(image):
        """Convert a path, PIL image, or [3,H,W] float tensor to RGB PIL."""
        if isinstance(image, (str, Path)):
            return Image.open(image).convert("RGB")
        if isinstance(image, Image.Image):
            return image.convert("RGB")
        if torch.is_tensor(image) and image.ndim == 3 and image.shape[0] == 3:
            array = (image.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
                     * 255).round().astype(np.uint8)
            return Image.fromarray(array).convert("RGB")
        raise TypeError("image must be a path, PIL image, or [3,H,W] tensor")

    @torch.inference_mode()
    def predict(self, image, prompt, yolo_conf=0.25):
        """Route a request, estimate selected-expert success, and dispatch or abstain."""
        pil_image = self._to_pil(image)
        resized = pil_image.resize((224, 224), Image.Resampling.BICUBIC)
        array = np.asarray(resized, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(self.device)
        output = self.router(image_tensor, [prompt])

        routing_probs = torch.softmax(
            output["routing_logits"] / self.temperature, dim=-1
        )[0]
        decision_id = int(routing_probs.argmax().item())
        routing_confidence = float(routing_probs.max().item())
        routing_probability_dict = {
            DECISION_NAMES[i]: float(routing_probs[i].item()) for i in range(3)
        }

        if decision_id == DECISION_CLARIFY:
            return {
                "status": "abstained", "route": "clarify",
                "reason": "Stage 2 gate identified an ambiguous request.",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": None, "result": None,
            }
        if routing_confidence < self.routing_threshold:
            return {
                "status": "abstained", "route": "clarify",
                "reason": "Stage 2 routing confidence is below its threshold.",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": None, "result": None,
            }

        success_probability = float(torch.sigmoid(
            output["quality_logits"][0, decision_id]
        ).item())
        if success_probability < self.quality_threshold:
            return {
                "status": "review_required", "route": DECISION_NAMES[decision_id],
                "reason": "Predicted expert success is below the configured quality threshold.",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": success_probability,
                "quality_threshold": self.quality_threshold,
                "result": None,
            }

        yolo_device = get_ultralytics_device(self.device)
        if decision_id == DECISION_CLASSIFICATION:
            result = self.cls_expert.predict(
                pil_image, device=yolo_device, conf=yolo_conf, verbose=False
            )[0]
            if result.probs is None:
                raise RuntimeError("Classification expert returned no class probabilities.")
            top_id = int(result.probs.top1)
            return {
                "status": "completed", "route": "classification",
                "routing_confidence": routing_confidence,
                "routing_probabilities": routing_probability_dict,
                "predicted_expert_success": success_probability,
                "top1_class_id": top_id,
                "top1_class_name": result.names[top_id],
                "top1_yolo_confidence": float(result.probs.top1conf.item()),
                "result": result,
            }

        result = self.seg_expert.predict(
            pil_image, device=yolo_device, conf=yolo_conf, verbose=False
        )[0]
        instance_count = 0 if result.masks is None else len(result.masks.data)
        return {
            "status": "completed", "route": "segmentation",
            "routing_confidence": routing_confidence,
            "routing_probabilities": routing_probability_dict,
            "predicted_expert_success": success_probability,
            "num_instances": instance_count,
            "has_masks": result.masks is not None,
            "result": result,
        }

print("Stage 3 inference wrapper defined.")

## 15. Save the Stage 3 checkpoint and configuration

The checkpoint contains the Stage 2 router parameters plus the newly trained quality head. The two YOLO expert checkpoints remain the Stage 1 files. The JSON configuration records the routing and quality thresholds used by this notebook.

In [ ]:
STAGE3_ROUTER_PATH = ARTIFACT_DIR / "router_performance_aware.pt"
STAGE3_CONFIG_PATH = ARTIFACT_DIR / "moe_performance_aware_config.json"

stage3_checkpoint = {
    "model_state_dict": router.state_dict(),
    "clip_model_name": router.clip_model_name,
    "clip_pretrained": router.clip_pretrained,
    "hidden_dim": 256,
    "dropout": 0.1,
    "temperature": STAGE2_TEMPERATURE,
    "routing_threshold": STAGE2_ROUTING_THRESHOLD,
    "quality_threshold": QUALITY_THRESHOLD,
    "segmentation_success_iou": SEGMENTATION_SUCCESS_IOU,
    "stage": 3,
}
torch.save(stage3_checkpoint, STAGE3_ROUTER_PATH)

stage3_config = {
    "architecture": "stage3_expert_performance_aware_multimodal_moe",
    "stage": 3,
    "router_checkpoint": STAGE3_ROUTER_PATH.name,
    "experts": {
        "classification": CLS_WEIGHTS_PATH.name,
        "segmentation": SEG_WEIGHTS_PATH.name,
    },
    "routing_temperature": STAGE2_TEMPERATURE,
    "routing_threshold": STAGE2_ROUTING_THRESHOLD,
    "quality_threshold": QUALITY_THRESHOLD,
    "quality_targets": {
        "classification": "top1_correct",
        "segmentation": f"foreground_iou >= {SEGMENTATION_SUCCESS_IOU}",
    },
    "quality_outputs_are_cross_task_comparable": False,
}
STAGE3_CONFIG_PATH.write_text(json.dumps(stage3_config, indent=2), encoding="utf-8")
print("Saved:", STAGE3_ROUTER_PATH)
print("Saved:", STAGE3_CONFIG_PATH)

## 16. Load the saved model and run example prompts

This cell reconstructs the Stage 3 model from the saved checkpoint and runs three example prompts. Depending on routing and quality confidence, a request may be completed, clarified, or marked for review.

In [ ]:
saved_checkpoint = torch.load(STAGE3_ROUTER_PATH, map_location=DEVICE)
loaded_router = CLIPPerformanceAwareRouter(
    clip_model_name=saved_checkpoint["clip_model_name"],
    clip_pretrained=saved_checkpoint["clip_pretrained"],
).to(DEVICE)
loaded_router.load_state_dict(saved_checkpoint["model_state_dict"], strict=True)
loaded_router.eval()

moe = PerformanceAwareYOLOMoE(
    router=loaded_router,
    cls_weights_path=CLS_WEIGHTS_PATH,
    seg_weights_path=SEG_WEIGHTS_PATH,
    temperature=saved_checkpoint["temperature"],
    routing_threshold=saved_checkpoint["routing_threshold"],
    quality_threshold=saved_checkpoint["quality_threshold"],
    device=DEVICE,
)

demo_image = quality_base[quality_eval_indices[0]][0].convert("RGB")
demo_prompts = [
    "Please classify the breed of this pet.",
    "Please segment the pet from the background.",
    "Can you help me understand this picture?",
]
for prompt in demo_prompts:
    prediction = moe.predict(demo_image, prompt)
    printable = {key: value for key, value in prediction.items() if key != "result"}
    print("\\nPrompt:", prompt)
    print(printable)

## 17. Visualize a Stage 3 segmentation result

The helper below overlays predicted masks only when the request was routed to segmentation and the expert was actually executed. Abstained/review-required requests are shown without a predicted mask.

In [ ]:
def show_stage3_prediction(image, prediction):
    """Display the input and any segmentation mask produced by Stage 3 inference.

    Inputs:
        image: path, PIL image, or supported RGB tensor.
        prediction: dictionary returned by PerformanceAwareYOLOMoE.predict().
    Output: matplotlib figure with input and prediction/review status.
    """
    pil_image = PerformanceAwareYOLOMoE._to_pil(image)
    image_array = np.asarray(pil_image)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(image_array)
    axes[0].set_title("Input image")
    axes[0].axis("off")
    axes[1].imshow(image_array)

    result = prediction.get("result")
    if (prediction.get("route") == "segmentation" and result is not None
            and result.masks is not None and len(result.masks.data) > 0):
        masks = result.masks.data.detach().cpu().numpy()
        combined = masks.max(axis=0) > 0.5
        combined = cv2.resize(
            combined.astype(np.uint8), (image_array.shape[1], image_array.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )
        axes[1].imshow(np.ma.masked_where(combined == 0, combined), alpha=0.5, cmap="cool")
        axes[1].set_title("Predicted segmentation")
    else:
        axes[1].set_title(f"No mask: {prediction.get('status')} / {prediction.get('route')}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

seg_prediction = moe.predict(demo_image, "Segment the pet from the background.")
show_stage3_prediction(demo_image, seg_prediction)
print({k: v for k, v in seg_prediction.items() if k != "result"})

## 18. What Stage 3 adds — and what it does not

### New functionality

- Learns from observed expert outcomes rather than treating the expert's raw confidence as the only quality signal.
- Predicts classification-expert success and segmentation-expert success separately.
- Adds an additional accept/review decision after Stage 2 routing.
- Reports task-specific quality prediction and selective-coverage metrics.

### Important limitations

- The two experts perform different tasks. This stage **does not compare them as interchangeable alternatives** and does not select the “better” task expert when the prompt asks for a specific task.
- Classification success is defined as top-1 correctness. Segmentation success is defined as IoU meeting `SEGMENTATION_SUCCESS_IOU`. Different thresholds or metrics will change the learned target.
- The quality head learns from outcomes generated by the existing YOLO checkpoints. It predicts success under this dataset and evaluation setup; it is not a universal estimate of model quality.
- The Oxford-IIIT Pet test split was already involved in Stage 1 checkpoint selection and Stage 2 calibration/evaluation. The held-out split in this notebook prevents source-image overlap between quality-head training and its evaluation subset, but it is not a pristine benchmark independent of all earlier development.
- The quality threshold is an operating choice. Review the coverage/success trade-off and do not assume `0.60` is optimal for every application.

## Suggested next experiments

1. Increase `MAX_QUALITY_IMAGES` and compare learning curves.
2. Change the segmentation IoU success criterion and observe how quality prediction changes.
3. Compare the Stage 2-only policy with the Stage 3 policy on the same held-out examples.
4. Collect expert outcomes on a new dataset that was not used during Stage 1 or Stage 2.
5. Add a human-review workflow for requests rejected by either routing or performance confidence.

## Stage 3 artifacts

After successful execution, the `artifacts` directory contains:

- `router_performance_aware.pt` — Stage 2 gate plus the trained expert-performance head.
- `moe_performance_aware_config.json` — model, expert, and threshold configuration.

The Stage 1 YOLO checkpoints remain unchanged and are reused by the Stage 3 inference wrapper.